# Lecture 2: Shallow and Deep Neural Networks


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Libraries loaded.")


## 1. Activation Functions

Prince's textbook places ReLU at the center. Let us compare different activation functions.


In [ ]:
x = np.linspace(-4, 4, 300)

relu    = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))
tanh_v  = np.tanh(x)
leaky   = np.where(x > 0, x, 0.1 * x)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, y, color) in zip(axes.flat,
        [("ReLU", relu, 'crimson'),
         ("Sigmoid", sigmoid, 'steelblue'),
         ("Tanh", tanh_v, 'darkorange'),
         ("Leaky ReLU (a=0.1)", leaky, 'purple')]):
    ax.plot(x, y, color=color, lw=2)
    ax.axhline(0, color='gray', lw=0.7, linestyle='--')
    ax.axvline(0, color='gray', lw=0.7, linestyle='--')
    ax.set_title(name, fontsize=13)
    ax.set_xlabel("x"); ax.set_ylabel("sigma(x)")
    ax.set_ylim(-1.5, 4.5)
    ax.grid(alpha=0.3)

plt.suptitle("Activation Functions", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_nb02_activations.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. Shallow Neural Network (Ch. 3)

A shallow network according to the textbook's formulation:

$$h_k = \\text{ReLU}(\\theta_{k0} + \\theta_{k1} x)$$
$$y = \\phi_0 + \\sum_{k=1}^{K} \\phi_k h_k$$

Each hidden neuron creates a **linear region**.  
Combining these regions allows approximation of complex functions.


In [ ]:
def shallow_net_1d(x, theta0, theta1, phi0, phi):
    # Shallow network with K hidden neurons
    # theta0, theta1: shape (K,)
    # phi0: scalar, phi: shape (K,)
    H = np.maximum(0, theta0[:, None] + theta1[:, None] * x[None, :])  # (K, N)
    return phi0 + phi @ H  # (N,)

x_plot = np.linspace(-3*np.pi, 3*np.pi, 500)
y_target = np.sin(x_plot)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, K in zip(axes, [4, 16, 64]):
    np.random.seed(7)
    theta0 = np.random.uniform(-3, 3, K)
    theta1 = np.random.uniform(-2, 2, K)
    phi0   = np.random.randn() * 0.1
    phi    = np.random.randn(K) * 0.5
    
    y_pred = shallow_net_1d(x_plot, theta0, theta1, phi0, phi)
    
    ax.plot(x_plot, y_target, 'gray', lw=1.5, alpha=0.6, label="Target: sin(x)")
    ax.plot(x_plot, y_pred, 'crimson', lw=1.5, label=f"Shallow net (K={K})")
    ax.set_title(f"{K} Hidden Neurons (random init)")
    ax.set_ylim(-4, 4); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Shallow Network Function Approximation (random weights)", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb02_shallow.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. Universal Approximation Theorem — Visual Proof

According to the theorem, a shallow network with sufficiently many hidden neurons can approximate **any continuous function** to arbitrary precision.  
Each ReLU neuron contributes a 'broken-line segment'; summing them yields complex shapes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x = np.linspace(-2, 2, 500)
colors = plt.cm.tab10(np.linspace(0, 1, 5))
individual = []
params = [(-1.5, 1, 0.8), (0, 1, -1.2), (0.5, 2, 0.6), (-0.8, -1, -0.5), (1.2, 1.5, 0.9)]
for i, (t0, t1, ph) in enumerate(params):
    h = np.maximum(0, t0 + t1*x)
    contribution = ph * h
    individual.append(contribution)
    axes[0].plot(x, contribution, color=colors[i], lw=1.5, label=f"phi{i+1}*ReLU()")

axes[0].set_title("Individual ReLU Contributions")
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[0].axhline(0, color='k', lw=0.5)

total = sum(individual)
axes[1].plot(x, total, 'crimson', lw=2.5, label="Total output")
axes[1].fill_between(x, total, alpha=0.15, color='crimson')
axes[1].set_title("Sum of Contributions -> Complex Function")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(0, color='k', lw=0.5)

plt.tight_layout()
plt.savefig("fig_nb02_universal.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Deep Neural Network (Ch. 4)

In deep networks, layers are **composed on top of each other**:

$$\\mathbf{h}_1 = a(\\mathbf{W}_1 \\mathbf{x} + \\mathbf{b}_1)$$
$$\\mathbf{h}_k = a(\\mathbf{W}_k \\mathbf{h}_{k-1} + \\mathbf{b}_k)$$
$$\\mathbf{y} = \\mathbf{W}_{K+1} \\mathbf{h}_K + \\mathbf{b}_{K+1}$$

Each layer transforms the input into a new **feature space**.


In [ ]:
def relu(x):
    return np.maximum(0, x)

def forward_pass(x, weights, biases):
    h = x.copy()
    for i, (W, b) in enumerate(zip(weights, biases)):
        z = h @ W.T + b
        h = relu(z) if i < len(weights)-1 else z
    return h

np.random.seed(13)
x_1d = np.linspace(-np.pi, np.pi, 300).reshape(-1, 1)
y_target_1d = np.sin(x_1d) * np.cos(2 * x_1d)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

configs = [
    ("Shallow: 1 layer, 100 neurons",
     [np.random.randn(100,1)*0.5, np.random.randn(1,100)*0.3],
     [np.random.randn(100)*0.1, np.random.randn(1)*0.1]),
    ("Deep: 4 layers, 25 neurons/layer",
     [np.random.randn(25,1)*0.5, np.random.randn(25,25)*0.3,
      np.random.randn(25,25)*0.3, np.random.randn(1,25)*0.3],
     [np.random.randn(25)*0.1, np.random.randn(25)*0.1,
      np.random.randn(25)*0.1, np.random.randn(1)*0.1])
]

for ax, (name, Ws, Bs) in zip(axes, configs):
    y_pred = forward_pass(x_1d, Ws, Bs).flatten()
    total_params = sum(W.size + b.size for W, b in zip(Ws, Bs))
    ax.plot(x_1d, y_target_1d, 'gray', lw=1.5, alpha=0.7, label="Target")
    ax.plot(x_1d, y_pred, 'steelblue', lw=2, label="Network prediction")
    ax.set_title(f"{name}\n(Total parameters: {total_params})")
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Shallow vs Deep Network (random init - no training)", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb02_deep_vs_shallow.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. Parameter Count Comparison

Let us compute the total number of parameters in a network:
- **Shallow**: input→H + H→output  
- **Deep**: each layer has its own W and b


In [ ]:
def count_params(layer_sizes):
    total = 0
    for i in range(len(layer_sizes)-1):
        W_params = layer_sizes[i] * layer_sizes[i+1]
        b_params = layer_sizes[i+1]
        total += W_params + b_params
    return total

architectures = {
    "Shallow [1,64,1]":          [1, 64, 1],
    "Deep    [1,16,16,16,16,1]": [1, 16, 16, 16, 16, 1],
    "Deep    [1,8,8,8,8,8,8,1]": [1, 8, 8, 8, 8, 8, 8, 1],
    "Wide    [1,128,1]":          [1, 128, 1],
    "Deep-Wide [1,32,32,32,1]":  [1,32,32,32,1],
}

print(f"{'Architecture':<35} {'Parameters':>12} {'Layers':>8}")
print("-"*57)
for name, layers in architectures.items():
    params = count_params(layers)
    depth = len(layers) - 1
    print(f"{name:<35} {params:>12,} {depth:>8}")


## 6. Summary

| Concept | Description |
|---|---|
| **Shallow network** | Single hidden layer; universal approximation possible but inefficient |
| **Deep network** | Multiple layers; exponential region growth, hierarchical representation |
| **ReLU** | Most common activation; easy gradient computation, less vanishing gradient |
| **Universal approximation** | Any continuous function can be approximated with enough neurons |
| **Depth advantage** | Much richer function space with the same number of parameters |
| **Forward pass** | Sequential matrix multiplications from input to output |

